<a href="https://colab.research.google.com/github/Usman-938/Management-System/blob/main/cusromer_churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part A and B

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- Part A: Create Dataset ---
np.random.seed(42)
n = 300
df = pd.DataFrame({
    "MonthlyUsageHours": np.random.normal(60, 20, n).clip(5, 120).round(1),
    "MonthlyBillPKR": np.random.normal(2500, 800, n).clip(500, 6000).round(0),
    "Complaints": np.random.poisson(1.2, n).clip(0, 6),
    "TenureMonths": np.random.randint(1, 60, n),
    "ContractType": np.random.choice(["Monthly", "Yearly"], n, p=[0.7, 0.3])
})

# Create churn label (Hidden logic: High bills and complaints = Churn)
score = (
    0.04*(df["MonthlyBillPKR"] - 2500) +
    0.8*(df["Complaints"]) -
    0.03*(df["TenureMonths"]) +
    1.2*(df["ContractType"] == "Monthly").astype(int) -
    0.02*(df["MonthlyUsageHours"] - 60)
)
prob = 1 / (1 + np.exp(-score/10))
df["Churn"] = (np.random.rand(n) < prob).astype(int)

print("Dataset Preview:")
print(df.head())

# --- Part B: Preprocess + Split ---
X = df.drop("Churn", axis=1)
y = df["Churn"]

cat_cols = ["ContractType"]
num_cols = ["MonthlyUsageHours", "MonthlyBillPKR", "Complaints", "TenureMonths"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("\nData splitting complete. Training size:", len(X_train))

Dataset Preview:
   MonthlyUsageHours  MonthlyBillPKR  Complaints  TenureMonths ContractType  \
0               69.9          1837.0           0            47      Monthly   
1               57.2          2052.0           3            43      Monthly   
2               73.0          3098.0           1            50       Yearly   
3               90.5          2988.0           1            23      Monthly   
4               55.3          2483.0           2            44      Monthly   

   Churn  
0      0  
1      0  
2      1  
3      1  
4      0  

Data splitting complete. Training size: 225


Part C

In [2]:
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Train Decision Tree
tree_model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", DecisionTreeClassifier(max_depth=4, random_state=42))
])

tree_model.fit(X_train, y_train)
pred_tree = tree_model.predict(X_test)

print("\n--- Decision Tree Results ---")
print("Accuracy:", accuracy_score(y_test, pred_tree))

# 2. Feature Importance
clf = tree_model.named_steps["clf"]
feature_names = tree_model.named_steps["prep"].get_feature_names_out()
imp = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=False)
print("\nTop Important Features:\n", imp)

# 3. Decision Path Explanation
def explain_tree_decision(pipeline, single_row):
    prep = pipeline.named_steps["prep"]
    clf = pipeline.named_steps["clf"]
    X_trans = prep.transform(single_row)
    node_indicator = clf.decision_path(X_trans)
    leave_id = clf.apply(X_trans)
    feature_names = prep.get_feature_names_out()

    print("\nDecision path (rules) for this customer:")
    for node_id in node_indicator.indices:
        if leave_id[0] == node_id: continue
        feature = clf.tree_.feature[node_id]
        threshold = clf.tree_.threshold[node_id]
        if feature != _tree.TREE_UNDEFINED:
            fname = feature_names[feature]
            val = X_trans[0, feature]
            direction = "LEFT" if val <= threshold else "RIGHT"
            print(f"- {fname} ({val:.2f}) vs Threshold ({threshold:.2f}) -> go {direction}")

    pred = pipeline.predict(single_row)[0]
    print("Final Prediction:", "Churn" if pred==1 else "No Churn")

# Pick one sample to explain
sample = X_test.sample(1, random_state=1)
explain_tree_decision(tree_model, sample)


--- Decision Tree Results ---
Accuracy: 0.88

Top Important Features:
 num__MonthlyBillPKR         0.857158
num__MonthlyUsageHours      0.092685
num__TenureMonths           0.026623
num__Complaints             0.020834
cat__ContractType_Yearly    0.002700
dtype: float64

Decision path (rules) for this customer:
- num__MonthlyBillPKR (2326.00) vs Threshold (2560.00) -> go LEFT
- num__MonthlyBillPKR (2326.00) vs Threshold (2114.00) -> go RIGHT
- num__MonthlyBillPKR (2326.00) vs Threshold (2158.50) -> go RIGHT
- num__MonthlyBillPKR (2326.00) vs Threshold (2316.00) -> go RIGHT
Final Prediction: No Churn


Part D

In [3]:
from sklearn.svm import SVC

# 1. Train SVM with RBF Kernel
svm_model = Pipeline(steps=[
    ("prep", preprocess),
    ("scale", StandardScaler(with_mean=False)),
    ("clf", SVC(kernel="rbf", C=5, gamma="scale", probability=True))
])

svm_model.fit(X_train, y_train)
pred_svm = svm_model.predict(X_test)

print("\n--- SVM (RBF) Results ---")
print("Accuracy:", accuracy_score(y_test, pred_svm))

# 2. Show Decision Confidence
sample2 = X_test.sample(1, random_state=7)
proba2 = svm_model.predict_proba(sample2)[0]
pred2 = svm_model.predict(sample2)[0]

print("\nCustomer for SVM check:")
print(sample2.to_string(index=False))
print(f"Prediction: {'Churn' if pred2==1 else 'No Churn'}")
print(f"Confidence (Probabilities): NoChurn={proba2[0]:.2f}, Churn={proba2[1]:.2f}")


--- SVM (RBF) Results ---
Accuracy: 0.8266666666666667

Customer for SVM check:
 MonthlyUsageHours  MonthlyBillPKR  Complaints  TenureMonths ContractType
              29.6          2770.0           2            39      Monthly
Prediction: Churn
Confidence (Probabilities): NoChurn=0.26, Churn=0.74
